# **음성인식 프로젝트**

1. 불 켜 / 불 꺼          1-1 / 1-2
2. 시작해 / 정지해        2-1 / 2-2
3. 앞으로 가 / 뒤로 가    3-1 / 3-2
4. 음악 틀어              4-1
5. 전화 받아              5-1
6. 모델 학습              6-1
7. 데이터 준비            7-1
8. 필터 적용              8-1

총 11개의 단어 학습

Whisper 모델을 기반으로 11개의 단어를 구별하기 위하여 Whisper 모델의 디코더를 잘라내고 가벼운 분류기(Classifier)을 달았음.

입력은 마이크를 들어온 3초간의 음성 데이터를 Log-Mel Spectrogram 을 통해 이미지 형태의 데이터로 바꿔냄.

Whisper Encoder 는 68만 시간 분량의 방대한 다국어 음성 데이터를 통해 학습됨 이를 통해 고차원적인 특징을 뽑아냄. -> 이 부분은 사전학습된 가중치를 전이학습을 통해 사용

인코더를 통해 나온 시계열 데이터를 평균내어 하나의 압축된 정보로 만드는 Mean Pooling 과정을 통해 Linear Layer (선형 분류기) 로 11개의 단어를 Classifier 함.

전처리는 아날로그 음성 신호를 16,000Hz로 샘플링하였고, STFT(Short Time Fourier Transform) 을 통해 멜 스펙트로그램으로 변환.

인코딩은 Whisper Encoder을 통한 트랜스포머의 핵심 메커니즘인 Self-Attention 이 동작함. 마지막으로 Linear Layer을 통해 분류를 하고 Softmax 함수를 통해 각 명령어일 확률을 계산함.

또한 Softmax를 통한 확률은 모델이 판단하는 각 라벨의 점수인 logits이 라벨별로 큰 차이가 없어 Softmax 를 통한 모델의 신뢰도를 판별하는 것이 비효율적이라고 생각해 학습이 끝난 모델에 Test 데이터를 넣어 중심값과 각 test 값 사이의 거리를 측정하고 99%가 속한 거리를 기준으로 이보다 더 바깥쪽에 있다면 알 수 없음으로 처리되고 90, 95, 99%의 신뢰기준을 기지고 입력된 데이터가 속해있는 지점에 따라 신뢰도 높음, 중간, 낮음 으로 구분될 수 있도록 하였습니다.

# **라이브러리 설치 및 데이터셋 불러오기**

In [ ]:
!pip install torchaudio librosa pandas scikit-learn tqdm --quiet
!pip install evaluate torchcodec -U transformers datasets accelerate --quiet
!pip install -q git+https://github.com/openai/whisper.git
!fc-cache -fv

In [ ]:
# ---------------------------------------------------------
# 1. 환경 설정: 라이브러리 설치 및 임포트
# ---------------------------------------------------------
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import tqdm
import librosa

# PyTorch 관련 임포트
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F

from torch.utils.data import DataLoader, Dataset

# Scikit-learn (평가 지표용)
from sklearn.metrics import classification_report, confusion_matrix

# Whisper 임포트
import whisper

from google.colab import drive

# **전처리 및 데이터 라벨링**

In [ ]:
# ---------------------------------------------------------
# 2. 구글 드라이브 마운트 및 경로 설정 (★중요★)
# ---------------------------------------------------------
print("📂 구글 드라이브를 연결합니다...")
drive.mount('/content/drive')

# 디바이스 설정
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"✅ 사용 중인 디바이스: {DEVICE}")

# ★ 아래 경로를 본인의 구글 드라이브 폴더 위치로 수정해주세요! ★
# 예: 내 드라이브(MyDrive) 안에 'dataset' 폴더가 있고 그 안에 'train', 'test'가 있는 경우
BASE_PATH = "/content/drive/MyDrive/Machine Learning/"
TRAIN_PATH = os.path.join(BASE_PATH, "Train Dataset")
TEST_PATH = os.path.join(BASE_PATH, "Test Dataset")

print(f"📌 학습 데이터 경로: {TRAIN_PATH}")
print(f"📌 테스트 데이터 경로: {TEST_PATH}")

# ---------------------------------------------------------
# 3. 라벨 매핑 및 설정
# ---------------------------------------------------------
LABEL_MAP = {
    '1-1': '불 켜', '1-2': '불 꺼',
    '2-1': '시작 해', '2-2': '정지 해',
    '3-1': '앞으로 가', '3-2': '뒤로 가',
    '4-1': '음악 틀어',
    '5-1': '전화 받아',
    '6-1': '모델 학습',
    '7-1': '데이터 준비',
    '8-1': '필터 적용',
}

ordered_labels = [
    "불 켜", "불 꺼", "시작 해", "정지 해",
    "앞으로 가", "뒤로 가", "음악 틀어",
    "전화 받아", "모델 학습", "데이터 준비", "필터 적용",
]

label_to_id = {label: i for i, label in enumerate(ordered_labels)}
id_to_label = {i: label for i, label in enumerate(ordered_labels)}
NUM_LABELS = len(ordered_labels)

# 오디오 설정
TARGET_SR = 16000
TARGET_SEC = 3.0
TARGET_SAMPLES = int(TARGET_SR * TARGET_SEC)  # 48000 samples

# 확인
print("label_to_id:", label_to_id)
print("id_to_label:", id_to_label)

class RawAudioDataset(Dataset):
    def __init__(self, data_df, target_length=48000):
        self.paths = data_df['path'].tolist()
        self.labels = data_df['label'].tolist()
        self.target_length = target_length

    def __len__(self):
        return len(self.paths)

    def __getitem__(self, idx):
        path = self.paths[idx]
        label = self.labels[idx]

        # 1. 오디오 로드 (Whisper load_audio는 16k 리샘플링 포함)
        audio = whisper.load_audio(path)

        # 2. 길이 맞추기 (Padding or Trimming) -> 고정 길이 48000
        if len(audio) > self.target_length:
            audio = audio[:self.target_length]
        else:
            audio = np.pad(audio, (0, self.target_length - len(audio)), mode='constant')

        # 3. Tensor 변환 (Float32)
        return torch.tensor(audio, dtype=torch.float32), torch.tensor(label, dtype=torch.long)


In [ ]:
def create_metadata(data_dir):
    rows = []
    if not os.path.exists(data_dir):
        return pd.DataFrame(rows)

    for fn in os.listdir(data_dir):
        if fn.lower().endswith(".wav"):
            base = fn.split("_")[0]  # '1-1_xxx.wav' -> '1-1'
            if base in LABEL_MAP:
                rows.append({
                    "path": os.path.join(data_dir, fn),
                    "label": label_to_id[LABEL_MAP[base]]
                })
    return pd.DataFrame(rows)

train_df = create_metadata(TRAIN_PATH)
test_df  = create_metadata(TEST_PATH)

print(f"Train 데이터: {len(train_df)}개")
print(f"Test 데이터: {len(test_df)}개")

if len(train_df) == 0:
    raise ValueError("⚠️ Train 데이터가 없습니다. TRAIN_PATH 경로를 확인하세요.")
if len(test_df) == 0:
    raise ValueError("⚠️ Test 데이터가 없습니다. TEST_PATH 경로를 확인하세요.")

# ------------------------------
# 5. Dataset / DataLoader (WhisperMelDataset만 사용)
# ------------------------------
TARGET_SR = 16000
TARGET_SEC = 3.0
TARGET_SAMPLES = int(TARGET_SR * TARGET_SEC)  # 48000


# ✅ 여기서부터는 WhisperMelDataset 쓰지 말고 RawAudioDataset만 사용하세요
train_dataset = RawAudioDataset(train_df, TARGET_SAMPLES)
test_dataset  = RawAudioDataset(test_df,  TARGET_SAMPLES)

train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True, num_workers=2)
test_loader  = DataLoader(test_dataset,  batch_size=16, shuffle=False, num_workers=2)

xb, yb = next(iter(train_loader))
print("raw batch:", xb.shape)   # [B, 48000]
print("label batch:", yb.shape) # [B]

class IntegratedWhisperFFT(nn.Module):
    def __init__(self, num_classes, whisper_size="small"):
        super().__init__()

        whisper_full = whisper.load_model(whisper_size)
        self.encoder = whisper_full.encoder

        for p in self.encoder.parameters():
            p.requires_grad = True

        d_model = self.encoder.conv2.out_channels  # small=768
        self.classifier = nn.Sequential(
            nn.Linear(d_model, 256),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(256, num_classes)
        )

        self.n_fft = 400
        self.hop = 160
        self.mel_bins = 80
        self.mel_len = 300

        filters = torch.tensor(
            whisper.audio.mel_filters(torch.device("cpu"), self.mel_bins),
            dtype=torch.float32
        )
        self.register_buffer("mel_filters", filters)
        self.register_buffer("window", torch.hann_window(self.n_fft))

        self.embedding_len = 150
        pe = self.encoder.positional_embedding
        self.encoder.positional_embedding = nn.Parameter(pe[:self.embedding_len, :])

    def audio_to_logmel(self, audio):
        """
        audio: [B,48000] 또는 [48000]
        return: [B,80,300]
        """
        if audio.dim() == 1:
            audio = audio.unsqueeze(0)  # [1,T]

        # ✅ Whisper 방식 reflect pad (200,200) : pad는 (N,C,L)에서 안전
        x = audio.unsqueeze(1)                 # [B,1,T]
        x = F.pad(x, (200, 200), mode="reflect")
        x = x.squeeze(1)                       # [B,T+400]

        stft = torch.stft(
            x,
            n_fft=self.n_fft,
            hop_length=self.hop,
            window=self.window.to(x.device),
            return_complex=True
        )  # [B, F, Frames]

        mag = stft.abs().pow(2)  # power

        mel_filters = self.mel_filters.to(x.device)     # [80, F]
        mel = torch.matmul(mel_filters, mag)            # [B,80,Frames]

        logmel = torch.clamp(mel, min=1e-10).log10()
        logmel = torch.maximum(logmel, logmel.max() - 8.0)
        logmel = (logmel + 4.0) / 4.0

        # 길이 300 고정
        T = logmel.size(-1)
        if T > self.mel_len:
            logmel = logmel[..., :self.mel_len]
        elif T < self.mel_len:
            logmel = F.pad(logmel, (0, self.mel_len - T))

        return logmel

    def forward_features(self, audio):
        logmel = self.audio_to_logmel(audio)   # [B,80,300]
        enc = self.encoder(logmel)             # [B,150,D]
        emb = enc.mean(dim=1)                  # [B,D]
        return emb

    def forward(self, audio):
        emb = self.forward_features(audio)
        logits = self.classifier(emb)
        return logits

# **모델 정의**

# **Whisper base Model**

In [ ]:
# ---------------------------------------------------------
# 5. 학습 루프 (raw audio)
# ---------------------------------------------------------
model = IntegratedWhisperFFT(NUM_LABELS, whisper_size="small").to(DEVICE)
print("✅ Raw->Mel->WhisperEncoder 모델 생성 완료")

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=1e-5)
EPOCHS = 8

train_losses = []
test_accs = []

print(f"\n🚀 학습 시작 (총 {EPOCHS} Epochs)")

for epoch in range(EPOCHS):
    model.train()
    running_loss = 0.0

    pbar = tqdm.tqdm(train_loader, desc=f"Epoch {epoch+1}/{EPOCHS}")
    for audio, labels in pbar:
        audio, labels = audio.to(DEVICE), labels.to(DEVICE)

        optimizer.zero_grad()
        logits = model(audio)
        loss = criterion(logits, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item()
        pbar.set_postfix({'loss': float(loss.item())})

    avg_loss = running_loss / max(1, len(train_loader))
    train_losses.append(avg_loss)

    # Test
    model.eval()
    correct, total = 0, 0
    with torch.no_grad():
        for audio, labels in test_loader:
            audio, labels = audio.to(DEVICE), labels.to(DEVICE)
            logits = model(audio)
            preds = logits.argmax(dim=1)
            correct += (preds == labels).sum().item()
            total += labels.size(0)

    acc = correct / total if total > 0 else 0
    test_accs.append(acc)
    print(f"   [Epoch {epoch+1}] Loss: {avg_loss:.4f} | Test Acc: {acc*100:.2f}%")


# ---------------------------------------------------------
# 6. 결과 시각화 및 혼동 행렬
# ---------------------------------------------------------
# Loss & Acc 그래프
plt.figure(figsize=(10, 4))
plt.subplot(1, 2, 1)
plt.plot(train_losses, label='Train Loss')
plt.title('Training Loss')
plt.legend()

plt.subplot(1, 2, 2)
plt.plot(test_accs, label='Test Accuracy', color='orange')
plt.title('Test Accuracy')
plt.legend()
plt.show()

# 최종 평가 (혼동 행렬)
model.eval()
all_preds = []
all_labels = []

with torch.no_grad():
    for audio, labels in test_loader:
        audio, labels = audio.to(DEVICE), labels.to(DEVICE)
        logits = model(audio)
        preds = logits.argmax(dim=1)

        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

print("\n[Final Classification Report]")
print(classification_report(all_labels, all_preds, target_names=ordered_labels, zero_division=0))

cm = confusion_matrix(all_labels, all_preds)
plt.figure(figsize=(8, 8))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=ordered_labels, yticklabels=ordered_labels)
plt.xlabel('Predicted')
plt.ylabel('True')
plt.title('Confusion Matrix (End-to-End Training)')
plt.show()

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

THRESHOLD = 0.125  # p99 (원하시는 값)

# ---------------------------------------------------------
# (1) Train embeddings 뽑기 (raw audio -> forward_features)
# ---------------------------------------------------------
@torch.no_grad()
def extract_embeddings_from_loader(model, loader, device, l2_normalize=True):
    model.eval()
    E_list, y_list = [], []
    for audio, labels in loader:
        audio = audio.to(device)
        labels = labels.to(device)

        emb = model.forward_features(audio)  # [B, D]
        if l2_normalize:
            emb = F.normalize(emb, p=2, dim=1)

        E_list.append(emb.detach().cpu())
        y_list.append(labels.detach().cpu())

    E = torch.cat(E_list, dim=0)  # [N, D]
    y = torch.cat(y_list, dim=0)  # [N]
    return E, y


def build_prototypes(E, y, num_classes):
    # E: [N,D]  (CPU 텐서)
    D = E.size(1)
    protos = torch.zeros(num_classes, D, dtype=E.dtype)
    for c in range(num_classes):
        idx = (y == c).nonzero(as_tuple=True)[0]
        if idx.numel() == 0:
            continue
        proto = E[idx].mean(dim=0)
        proto = F.normalize(proto, p=2, dim=0)
        protos[c] = proto
    return protos


# ---------------------------------------------------------
# (2) Unknown wrapper (raw audio 입력 / 튜플 3개 출력)
#  - pred_out: Long [B]  (unknown이면 -1)
#  - conf:     Float[B]  (softmax top1 prob)
#  - dmin:     Float[B]  (cosine distance min)
# ---------------------------------------------------------
class ModelWithUnknownRaw(nn.Module):
    def __init__(self, base_model, protos, threshold=0.125):
        super().__init__()
        self.base = base_model
        self.register_buffer("protos", protos)  # [C, D] (CPU/저장됨)
        self.threshold = float(threshold)

    def forward(self, audio):
        # logits -> softmax top1 prob
        logits = self.base(audio)                 # [B, C]
        probs = torch.softmax(logits, dim=1)      # [B, C]
        conf, pred = torch.max(probs, dim=1)      # [B], [B]

        # embedding -> cosine distance to prototypes
        emb = self.base.forward_features(audio)   # [B, D]
        emb = F.normalize(emb, p=2, dim=1)

        cos = torch.matmul(emb, self.protos.t())  # [B, C]
        dist = 1.0 - cos                          # cosine distance
        dmin, _ = torch.min(dist, dim=1)          # [B]

        # unknown이면 -1로 치환
        pred_out = pred.clone()
        unknown_mask = dmin > self.threshold
        pred_out = torch.where(
            unknown_mask,
            torch.full_like(pred_out, -1),
            pred_out
        )
        return pred_out, conf, dmin


# ---------------------------------------------------------
# (3) Export: trace -> optimize -> _save_for_lite_interpreter
# ---------------------------------------------------------
def export_lite_ptl(model, train_loader, num_labels, threshold, out_name="whisper_kws_mobile.ptl"):
    device = next(model.parameters()).device

    # 1) prototype 계산은 CPU에서 하는게 안전/가벼움
    train_E, train_y = extract_embeddings_from_loader(model, train_loader, device, l2_normalize=True)
    protos = build_prototypes(train_E, train_y, num_labels)  # CPU

    # 2) export는 CPU로 내리기(모바일이니까)
    base_cpu = model.to("cpu").eval()
    wrapped = ModelWithUnknownRaw(base_cpu, protos, threshold=threshold).to("cpu").eval()

    # 3) trace 입력 예시 (반드시 [1,48000]!)
    example_audio = torch.zeros(1, 48000, dtype=torch.float32)

    with torch.no_grad():
        traced = torch.jit.trace(wrapped, example_audio, strict=False)
        traced = torch.jit.freeze(traced)
        traced_opt = torch.jit.optimize_for_inference(traced)

        # ✅ Lite 전용 저장 (bytecode.pkl 생성)
        traced_opt._save_for_lite_interpreter(out_name)

    print(f"✅ Saved Lite PTL: {out_name}")
    return out_name


# ---- 실행 ----
ptl_name = export_lite_ptl(
    model=model,                 # 학습 완료된 IntegratedWhisperFFT
    train_loader=train_loader,   # raw audio 로더
    num_labels=NUM_LABELS,
    threshold=THRESHOLD,
    out_name="whisper_kws_mobile.ptl"  # 앱 assets에 넣을 파일명 그대로 추천
)


In [ ]:
import os
print("파일 존재:", os.path.exists("whisper_kws_mobile.ptl"))
print("파일 크기(bytes):", os.path.getsize("whisper_kws_mobile.ptl"))


In [ ]:
with torch.no_grad():
    pred_out, conf, dmin = traced(example_audio)

print("pred_out:", pred_out.shape, pred_out.dtype)
print("conf    :", conf.shape, conf.dtype)
print("dmin    :", dmin.shape, dmin.dtype)
print("values  :", pred_out.cpu().numpy(), conf.cpu().numpy(), dmin.cpu().numpy())

In [ ]:
import zipfile

path = "whisper_kws_mobile.ptl"
with zipfile.ZipFile(path, "r") as z:
    names = z.namelist()
    print("Contains bytecode.pkl ?", any("bytecode.pkl" in n for n in names))
    print("Top entries:", names[:20])


# **거리 확인**

In [ ]:
from sklearn.metrics import confusion_matrix, classification_report
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np

@torch.no_grad()
def eval_with_unknown(wrapped_model, loader, device):
    wrapped_model.eval()
    y_true, y_pred = [], []

    for audio, labels in loader:
        audio = audio.to(device)
        labels = labels.to(device)

        pred_out, conf, dmin = wrapped_model(audio)  # pred_out: -1 가능
        y_true.extend(labels.cpu().numpy().tolist())
        y_pred.extend(pred_out.cpu().numpy().tolist())

    return np.array(y_true), np.array(y_pred)

y_true, y_pred = eval_with_unknown(wrapped, test_loader, DEVICE)

# Unknown(-1) 포함해서 label list 확장
labels_ext = ordered_labels + ["Unknown"]

# confusion matrix용으로 -1을 마지막 인덱스로 매핑
y_pred_cm = np.where(y_pred == -1, NUM_LABELS, y_pred)

cm = confusion_matrix(y_true, y_pred_cm, labels=list(range(NUM_LABELS+1)))

plt.figure(figsize=(9,8))
sns.heatmap(
    cm, annot=True, fmt="d", cmap="Blues",
    xticklabels=labels_ext, yticklabels=labels_ext[:-1] + ["(True has no Unknown)"]
)
plt.xlabel("Predicted")
plt.ylabel("True")
plt.title("Confusion Matrix (with Unknown)")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

print(classification_report(
    y_true, np.clip(y_pred_cm, 0, NUM_LABELS),
    target_names=labels_ext,
    zero_division=0
))


In [ ]:
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
import librosa
import whisper
import matplotlib.pyplot as plt
import seaborn as sns
from torch.utils.data import Dataset, DataLoader
from torch.nn.utils.rnn import pad_sequence

# ----------------------------------------
# 0) 설정
# ----------------------------------------
TARGET_SR = 16000
TARGET_SEC = 3.0
TARGET_LEN = int(TARGET_SR * TARGET_SEC)

# ----------------------------------------
# 1) Whisper mel Dataset (항상 mel 반환)
# ----------------------------------------
class WhisperMelDataset(Dataset):
    def __init__(self, df, target_len=TARGET_LEN, top_db=35):
        self.df = df.reset_index(drop=True)
        self.target_len = target_len
        self.top_db = top_db

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]

        # path / label은 확실히 scalar로
        path = str(row["path"])
        label = int(row["label"])

        # 1) load audio (16kHz, float32)
        audio = whisper.load_audio(path)  # 1D float32

        # 2) trim silence (optional)
        audio_trim, _ = librosa.effects.trim(audio, top_db=self.top_db)
        if len(audio_trim) < TARGET_SR * 0.3:   # 너무 짧아지면 백업
            audio_trim = audio

        # 3) pad/cut to 3s
        if len(audio_trim) >= self.target_len:
            audio_fixed = audio_trim[:self.target_len]
        else:
            audio_fixed = np.pad(audio_trim, (0, self.target_len - len(audio_trim)), mode="constant")

        # 4) log-mel (torch.Tensor [80, T])
        mel = whisper.log_mel_spectrogram(audio_fixed)  # torch.Tensor [80,T]

        return mel, label

# ----------------------------------------
# 2) collate_fn: [80,T] 들을 [B,80,Tmax]로 패딩
# ----------------------------------------
def collate_mels(batch):
    mels, labels = zip(*batch)  # list of [80,T], list of int

    # pad_sequence는 [T,80] 형태가 편해서 transpose 후 padding
    mels_T80 = [m.T for m in mels]  # [T,80]
    mels_pad = pad_sequence(mels_T80, batch_first=True)  # [B,Tmax,80]
    mels_pad = mels_pad.permute(0, 2, 1).contiguous()    # [B,80,Tmax]

    labels = torch.tensor(labels, dtype=torch.long)
    return mels_pad, labels

# ----------------------------------------
# 3) DataLoader 생성
# ----------------------------------------
train_dataset = WhisperMelDataset(train_df)
test_dataset  = WhisperMelDataset(test_df)

train_loader = DataLoader(train_dataset, batch_size=8, shuffle=False, collate_fn=collate_mels)
test_loader  = DataLoader(test_dataset,  batch_size=8, shuffle=False, collate_fn=collate_mels)

# (선택) shape sanity check
xb, yb = next(iter(train_loader))
print("Sanity check - mel batch:", xb.shape, "labels:", yb.shape)  # [B,80,T]

# ----------------------------------------
# 4) embedding 추출 (IntegratedWhisperFFT: model.encoder 사용)
# ----------------------------------------
def extract_embeddings(model, loader, device, l2_normalize=True):
    model.eval()
    embs, labels = [], []

    with torch.no_grad():
        for mel, y in loader:
            # mel must be [B,80,T]
            if mel.ndim != 3:
                raise ValueError(f"mel ndim must be 3, got {mel.ndim}, shape={mel.shape}")

            # 혹시 [B,T,80]로 들어오면 자동 교정
            if mel.shape[1] != 80 and mel.shape[2] == 80:
                mel = mel.permute(0, 2, 1).contiguous()

            if mel.shape[1] != 80:
                raise ValueError(f"mel channel must be 80, got shape={mel.shape}")

            mel = mel.to(device)
            y = y.to(device)

            # ✅ 핵심: IntegratedWhisperFFT에는 whisper가 아니라 encoder가 있다
            enc = model.encoder(mel)      # 보통 [B, T', D]
            if enc.ndim != 3:
                raise ValueError(f"encoder output must be 3D [B,T,D], got {enc.shape}")

            pooled = enc.mean(dim=1)      # [B,D]

            if l2_normalize:
                pooled = F.normalize(pooled, p=2, dim=1)

            embs.append(pooled.cpu().numpy())
            labels.append(y.cpu().numpy())

    E = np.concatenate(embs, axis=0).astype(np.float32)
    y = np.concatenate(labels, axis=0).astype(int)
    return E, y

# ----------------------------------------
# 5) prototype 생성 & 거리 계산 함수들
# ----------------------------------------
def build_prototypes(E, y, num_labels):
    """
    E: (N,D), y:(N,)
    return protos: (K,D) 각 클래스 평균
    """
    D = E.shape[1]
    protos = np.zeros((num_labels, D), dtype=np.float32)
    for k in range(num_labels):
        idx = np.where(y == k)[0]
        if len(idx) == 0:
            protos[k] = 0
        else:
            protos[k] = E[idx].mean(axis=0)
            # prototype도 normalize하면 cosine 거리 안정적
            norm = np.linalg.norm(protos[k]) + 1e-9
            protos[k] = protos[k] / norm
    return protos

def cosine_distance(a, b):
    # a:(D,), b:(D,) normalized 가정하면 1 - dot
    return 1.0 - float(np.dot(a, b))

def compute_distances(E, y, protos, metric="cosine"):
    """
    E:(N,D), y:(N,), protos:(K,D)
    returns:
      d_min: (N,) nearest proto distance
      d_true:(N,) distance to true proto
      nn:   (N,) nearest proto index
    """
    N = E.shape[0]
    K = protos.shape[0]
    d_all = np.zeros((N, K), dtype=np.float32)

    if metric == "cosine":
        # protos, E가 normalize 되어있다면 dot 기반 빠르게 가능
        # d = 1 - (E @ protos.T)
        sims = E @ protos.T  # (N,K)
        d_all = 1.0 - sims
    else:
        raise ValueError("metric should be 'cosine'")

    nn = d_all.argmin(axis=1)
    d_min = d_all[np.arange(N), nn]
    d_true = d_all[np.arange(N), y]
    return d_min, d_true, nn

# ----------------------------------------
# 6) 실행: embeddings -> prototypes -> d_min 분석
# ----------------------------------------
train_E, train_y = extract_embeddings(model, train_loader, DEVICE, l2_normalize=True)
test_E,  test_y  = extract_embeddings(model, test_loader,  DEVICE, l2_normalize=True)

print("train_E:", train_E.shape, "test_E:", test_E.shape)

protos = build_prototypes(train_E, train_y, NUM_LABELS)
print("protos:", protos.shape)

train_dmin, train_dtrue, train_nn = compute_distances(train_E, train_y, protos, metric="cosine")
test_dmin,  test_dtrue,  test_nn  = compute_distances(test_E,  test_y,  protos, metric="cosine")

# ----------------------------------------
# 7) 분포 plot + 임계값 후보 출력
# ----------------------------------------
plt.figure(figsize=(9,5))
sns.histplot(train_dtrue, bins=30, kde=True, label="Train: d_true(to true proto)", stat="density")
sns.histplot(test_dmin,  bins=30, kde=True, label="Test:  d_min(to nearest proto)", stat="density")
plt.title("Distance Distribution (cosine distance = 1 - cos_sim)")
plt.xlabel("distance")
plt.legend()
plt.tight_layout()
plt.show()

# 임계값 후보: train_dtrue의 상위 퍼센타일 기반
pcts = [90, 95, 97, 99]
thr = np.percentile(train_dtrue, pcts)
print("\n[Threshold candidates from Train d_true percentiles]")
for p, t in zip(pcts, thr):
    print(f"  p{p:02d}: {t:.4f}")

# ----------------------------------------
# 8) Test 샘플별 테이블 (d_min 큰 순)
# ----------------------------------------
test_pred_by_dist = test_nn
test_correct_by_dist = (test_pred_by_dist == test_y)

df_dist = pd.DataFrame({
    "true": test_y,
    "true_name": [id_to_label[i] for i in test_y],
    "nn": test_pred_by_dist,
    "nn_name": [id_to_label[i] for i in test_pred_by_dist],
    "d_min": test_dmin,
    "d_true": test_dtrue,
    "correct_nn": test_correct_by_dist
})

display(df_dist.sort_values("d_min", ascending=False).head(30))


# **데이터 분석**

In [ ]:
# 1) 나눔고딕 TTF 다운로드
!wget -O /content/NanumGothic.ttf \
  https://github.com/google/fonts/raw/main/ofl/nanumgothic/NanumGothic-Regular.ttf


In [ ]:
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm

font_path = "/content/NanumGothic.ttf"

# 2-1) 폰트 매니저에 수동 등록
fm.fontManager.addfont(font_path)

# 2-2) 이 파일에서 폰트 이름 읽어오기
font_name = fm.FontProperties(fname=font_path).get_name()
print("등록된 폰트 이름:", font_name)

# 2-3) Matplotlib 기본 폰트로 설정
plt.rcParams["font.family"] = font_name
plt.rcParams["axes.unicode_minus"] = False  # 음수 부호 깨짐 방지


In [ ]:
import numpy as np

def collect_mfcc_from_dataset(dataset):
    """
    MFCCDataset에서 (x, label)을 꺼내서
    x: [1, 40, T] -> [40, T] numpy 로 바꿔 모읍니다.
    """
    mfcc_list = []
    label_list = []

    for x, y in dataset:
        # x: [1, n_mfcc, T]
        mfcc = x.squeeze(0).numpy()  # [40, T]
        mfcc_list.append(mfcc)
        label_list.append(int(y))

    mfcc_arr = np.stack(mfcc_list)       # (N, 40, T)
    labels_arr = np.array(label_list)    # (N,)
    return mfcc_arr, labels_arr

train_mfcc, train_labels = collect_mfcc_from_dataset(train_dataset)
eval_mfcc,  eval_labels  = collect_mfcc_from_dataset(eval_dataset)

print("train_mfcc shape:", train_mfcc.shape)   # (N_train, 40, T)
print("eval_mfcc  shape:", eval_mfcc.shape)    # (N_eval,  40, T)


In [ ]:
NUM_LABELS = len(unique_labels)

label_templates = {}

for lab in range(NUM_LABELS):
    mfccs = train_mfcc[train_labels == lab]  # (N_lab, 40, T)
    template = mfccs.mean(axis=0)            # (40, T)
    label_templates[lab] = template

print("라벨별 템플릿 생성 완료!")


In [ ]:
from numpy.linalg import norm

def cosine_sim(a, b):
    return np.dot(a.flatten(), b.flatten()) / (norm(a.flatten()) * norm(b.flatten()) + 1e-9)

# 라벨 내 평균 유사도
label_self_sim = {}

for lab in range(NUM_LABELS):
    mfccs = train_mfcc[train_labels == lab]
    sims = [cosine_sim(m, label_templates[lab]) for m in mfccs]
    label_self_sim[lab] = np.mean(sims)

print("\n라벨 내 평균 유사도:")
for lab, sim in label_self_sim.items():
    print(f"{id_to_label[lab]}: {sim:.3f}")


In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

# 라벨 간 유사도 행렬
sim_matrix = np.zeros((NUM_LABELS, NUM_LABELS))

for i in range(NUM_LABELS):
    for j in range(NUM_LABELS):
        sim_matrix[i, j] = cosine_sim(label_templates[i], label_templates[j])

plt.figure(figsize=(10, 8))
sns.heatmap(sim_matrix, annot=True, fmt=".2f", cmap="Blues",
            xticklabels=unique_labels, yticklabels=unique_labels)
plt.title("라벨 간 MFCC Prototype 유사도 Heatmap")
plt.show()


In [ ]:
from sklearn.manifold import TSNE

# MFCC를 (N, 40*T) 형태로 펼침
N, F, T = train_mfcc.shape
flattened = train_mfcc.reshape(N, -1)

# t-SNE 적용
tsne = TSNE(n_components=2, random_state=42, perplexity=10)
embedded = tsne.fit_transform(flattened)

plt.figure(figsize=(8, 6))
for lab in range(NUM_LABELS):
    idx = (train_labels == lab)
    plt.scatter(embedded[idx, 0], embedded[idx, 1], s=40, alpha=0.7, label=id_to_label[lab])

plt.title("MFCC 기반 t-SNE 시각화 (Train Set)")
plt.legend()
plt.show()


In [ ]:
for lab in range(NUM_LABELS):
    plt.figure(figsize=(10, 4))
    sns.heatmap(label_templates[lab], cmap="magma")
    plt.title(f"MFCC 평균 패턴 - {id_to_label[lab]}")
    plt.xlabel("Time")
    plt.ylabel("MFCC Coefficients")
    plt.show()
